In [16]:
from gurobipy import *
import numpy as np
import datetime as dt
from gurobipy import Model, GRB, quicksum
from collections import defaultdict

### Input dataset and set backhaul rate

In [ ]:
def mfsolver(file, max_c, num_b_cus):
    """
    Solve the MF-EVRPTW problem.
    
    Args:
        file (str): Name of the instance file.
        max_c (int): Maximum number of customers.
        num_b_cus (int): Backhaul rate denominator for pickup customers.
    """
    # Read data from file
    backnum = num_b_cus  # 2 is 50% 3 is 66% 4 is 80%

    num_s, num_c, num_depot = 0, 0, 0
    xc, yc, q, e, l, s = [], [], [], [], [], []

    # read data
    with open("E-VRPTW Instances/C100/"+ file + ".txt") as f:
        meet_empty_line = False
        for line in f.readlines()[1:]:
            # print(f"line[0]:{line[0]}")
            
            if line == "\n":
                meet_empty_line = True
                continue
            if not meet_empty_line:
                if line[0] in "DCS":
                    # instance information
                    name, type_, x, y, demand, ready_time, due_time, service_time = line.split()
                    # set dummy vertices of set of stations
                    # repeat_count = 1 if type_ == 'f' else 1
                    # for _ in range(repeat_count):
                    if num_c < max_c:
                        
                        xc.append(float(x))
                        yc.append(float(y))
                        q.append(float(demand))
                        e.append(float(ready_time))
                        l.append(float(due_time))
                        s.append(float(service_time))
                    
                        if type_ == "f":
                            num_s += 1
                        elif type_ == "c":
                            num_c += 1
                        else:
                            num_depot += 1
                else:
                    raise Exception("wrong file")
            else:
                # vehicle information
                end_num = float(line[line.find("/")+1:-2])
                if line[0] == "Q": # Battery capacity
                    Q = end_num
                elif line[0] == "C": # Cargo capacity
                    C = end_num
                elif line[0] == "r": # Charge consumption rate
                    h = end_num
                elif line[0] == "g": # Charge rate
                    g = end_num
                elif line[0] == "v": # Speed
                    Speed = end_num
                else:
                    raise Exception("wrong file")
        # set dummy vertices of depot
        xc.append(xc[0])
        yc.append(yc[0])
        q.append(q[0])
        e.append(e[0])
        l.append(l[0])
        s.append(s[0])

    print(num_s, num_c, Speed, xc, yc, q)

    """set"""
    depot_0 = [0]
    depot_n = [num_s + num_c + 1]
    print(depot_n)

    stations = [i for i in range(1, num_s+1)]
    print('s', stations)
    stations_0 = depot_0 + stations

    customers = [i for i in range(num_s+1, num_s+num_c+1)]
    print('c', customers)
    customers_0 = depot_0 + customers
    customers_n = customers + depot_n

    stations_customers = stations + customers
    stations_customers_0 = depot_0 + stations_customers
    stations_customers_n = stations_customers + depot_n
    total = depot_0 + stations + customers + depot_n
    print(total)

    BinDD = []
    BinPP = []

    pidx = 1
    for idx, i in enumerate(customers):
        if pidx % backnum == 0:
            BinPP.append(i)
            
        else:
            BinDD.append(i)   
            
        pidx += 1

    for i in BinPP:
        q[i] *= -1


    model = Model("MFVRPTW_PR")

    # Define the mixed fleet.
    # E: Battery Electric Truck (BET)
    # G: Diesel Truck (DT)
    #
    # The field "cap" denotes the energy capacity used in the routing model.
    # - BET: cap = battery capacity (Q).
    # - DT: cap = 10000, a sufficiently large value used to emulate unlimited
    #   driving range, since diesel trucks are not subject to battery constraints.
    fleet = [
        {"id": "A1", "type": "E", "cap": Q},
        {"id": "A2", "type": "E", "cap": Q},
        {"id": "B1", "type": "G", "cap": 10000},
        {"id": "B2", "type": "G", "cap": 10000},
    ]

    V = list(range(len(fleet)))
    EV = [v for v in V if fleet[v]["type"] == "E"]
    GV = [v for v in V if fleet[v]["type"] == "G"]

    """arcs and distance/time matrix"""
    A = [(v, i, j) for v in V for i in stations_customers_0 for j in stations_customers_n]
    Dist = {(i, j): np.hypot(xc[i]-xc[j], yc[i]-yc[j]) for _, i, j in A} # distance matrix
    Time = {(i, j): (np.hypot(xc[i]-xc[j], yc[i]-yc[j]))/Speed for _, i, j in A} # time matrix

    """decision var"""
    t = model.addVars(total, vtype = GRB.CONTINUOUS, name = "arrive time")
    u = model.addVars(total, vtype = GRB.CONTINUOUS, name = "remaining cargo")
    # battery level (only relevant for EVs)
    b = {(v, i): model.addVar(lb=0.0, name=f"b_{v}_{i}")
        for v in EV for i in total}

    x = model.addVars(A, vtype = GRB.BINARY, name = "arcs")
    z = model.addVars(stations, vtype=GRB.BINARY, name="station visits")

    """objective"""
    # Total travel distance (vehicle-arc-specific, but same Dist(i,j) used)
    dist_cost = quicksum(Dist[i, j] * x[v, i, j] for v, i, j in A)
    # Optional penalty for unused customers or infeasibilities (e.g., leaving depot)
    penalty = 1000 * quicksum(x[v, i, j] for v, i, j in A if i == 0)

    # Set total objective
    obj = dist_cost + penalty
    model.setObjective(obj, GRB.MINIMIZE)

        # each customer is visited exactly once
    model.addConstrs((quicksum(x[v, i, j] for v in EV for j in stations_customers_n if i!=j) + (quicksum(x[v, i, j] for v in GV for j in customers_n if i!=j))  == 1 for i in customers), name = "c0")

    for j in stations:
            model.addConstr(quicksum(x[v, i, j] for v in EV for i in stations_customers_0 if i != j) <= 1000 * z[j], name=f"station_visit_link_{j}")
        
    model.addConstr(quicksum(z[j] for j in stations) <= 1, name="max_one_station_visit")

    for v in EV:
        model.addConstr(quicksum(x[v, 0, j] for j in stations_customers if j != 0) <= 1, f"depart_{v}")
        # model.addConstr(quicksum(x[v, j, 0] for j in stations_customers if j != 0) <= 1, f"return_{v}")

    for v in GV:
        model.addConstr(quicksum(x[v, 0, j] for j in customers if j != 0) <= 1, f"depart_{v}")
        # model.addConstr(quicksum(x[v, j, 0] for j in customers if j != 0) <= 1, f"return_{v}")
    
    # handle the connectivity of visits to stations
    model.addConstrs((quicksum(x[v, i, j] for j in stations_customers_n if i!=j) <= 1 for v in EV for i in stations), name = "c1")

    # after a vehicle arrives at a customer or station it has to leave for another destination
    for v in EV:
        model.addConstrs((quicksum(x[v,j,i] for i in stations_customers_n if i!=j) - quicksum(x[v, i,j] for i in stations_customers_0 if i!=j) == 0  for j in stations_customers), "c2_ev")

    # after a vehicle arrives at a customer or station it has to leave for another destination
    for v in GV:
        model.addConstrs((quicksum(x[v,j,i] for i in customers_n if i!=j) - quicksum(x[v, i,j] for i in customers_0 if i!=j) == 0  for j in customers), "c2_gv")

    # time feasibility for arcs leaving customers and depots
    model.addConstrs((t[i] + (Time[i, j] + s[i])* x[v, i, j] - l[0]*(1- x[v, i, j]) <= t[j] for v in V for i in customers_0 for j in stations_customers_n if i!=j), name = "c3")

    # time feasibility for arcs leaving stations
    model.addConstrs((t[i] + Time[i, j]*x[v, i, j] + g*(Q-b[v, i]) - (l[0]+Q*g)*(1-x[v, i, j]) <= t[j] for v in EV for i in stations for j in stations_customers_n if i!=j), name = "c4")

    # every node is visited within the time window
    model.addConstrs((t[i] >= e[i] for i in total),name = "tw_early")
    model.addConstrs((t[i] <= l[i] for i in total), name = "tw_due")

    # demand fulfillment at all customers
    for v1 in EV:
        for v2 in GV:
            model.addConstrs((u[j] <= u[i] -q[i]* (x[v1,i,j] + x[v2,i,j]) + C*(1-x[v1, i, j]-x[v2,i,j]) for i in stations_customers_0 for j in stations_customers_n if i!=j), name = "c7")
    model.addConstrs((u[i] <= C for i in depot_0), name = "c8")
    model.addConstrs((u[i] >= 0 for i in total), name = "c9")

    # battery feasibility for arcs leaving customers
    model.addConstrs((b[v, j] <= b[v, i] - (h*Dist[i, j])*x[v,i,j] + Q*(1-x[v, i, j]) for v in EV for i in customers for j in stations_customers_n if i!=j), name = "c9")
    # battery feasibility for arcs leaving stations and depots
    model.addConstrs((b[v, j] <= Q - (h*Dist[i, j])*x[v, i,j] for v in EV for i in stations_0 for j in stations_customers_n if i!=j), name = "c10") # update 0505 revised from S_paper
    model.addConstrs((b[v, i] >= 0 for v in EV for i in total), name = "c11")  # update 0504 add the battery constraints

    ### Backhaul constraints
    for v in V:
        for i in BinPP:
            for j in BinDD:
                model.addConstr(x[v,i,j] == 0, name=f"no_pickup_to_delivery_{i}_{j}")

    ### Backhaul constraints
    for v in V:
        for p in BinPP:
            model.addConstr(x[v,0,p] == 0, name=f"no_pickup_to_delivery_{i}_{j}")
            
    for v in EV:
        for p in BinPP:
            for j in stations:
                for d in BinDD:
                    model.addConstr(x[v,p,j] + x[v,j,d] <= 1)

    for v in EV:         
        for p in depot_0:
            for j in stations:
                for d in BinPP:
                    model.addConstr(x[v,p,j] + x[v,j,d] <= 1)


    """params"""
    model.Params.TimeLimit = 100

    """optimize"""
    model.update()
    model.Params.OutputFlag = 0
    model.optimize()

    print(f"Status:{model.Status}")
    print(f"Objective:{model.ObjVal}")
    print(f"CPU time:{model.Runtime}")

    """routes"""
    optimal_x = model.getAttr('X', x)

    connections = []

    for key, value in optimal_x.items():
        if value == 1:
            connections.append(key)

    # group arcs by vehicle
    routes = defaultdict(list)
    for v, i, j in connections:
        routes[v].append((i, j))

    # reconstruct ordered routes
    decoded_routes = {}
    for v, arcs in routes.items():
        # build mapping of start->end
        next_node = {i: j for i, j in arcs}
        # find starting node (the one that’s not an end of any other arc)
        all_starts = set(next_node.keys())
        all_ends = set(next_node.values())
        start = list(all_starts - all_ends)
        if not start:
            start = [0]  # fallback to depot
        start = start[0]

        # follow chain
        route = [start]
        while start in next_node:
            start = next_node[start]
            route.append(start)
        decoded_routes[v] = route

    print(decoded_routes)
    print(f'Gurobi solution: The total distance : {round(dist_cost.getValue(), 2)}')
    print(f"CPU time:{round(model.Runtime, 2)} seconds")
    print('DP list', BinDD, BinPP)
    print('file name: ', file, 'backhaul number is 2:50%; 3:66%; 4:80%:', backnum)
    print(f'num_s is {num_s}, num_c is {num_c}, current time is {dt.datetime.now().strftime("%m%d-%H%M")}')

In [18]:
filename = 'c101_21'
num_cus = 5
num_b_cus = 3

mfsolver(filename, num_cus, num_b_cus)

21 5 1.0 [40.0, 40.0, 73.0, 90.0, 55.0, 69.0, 32.0, 39.0, 23.0, 4.0, 18.0, 1.0, 13.0, -5.0, 26.0, 28.0, 39.0, 48.0, 59.0, 73.0, 74.0, 90.0, 45.0, 45.0, 42.0, 42.0, 42.0, 40.0] [50.0, 50.0, 52.0, 55.0, 79.0, 89.0, 80.0, 96.0, 80.0, 83.0, 58.0, 56.0, 31.0, 38.0, 37.0, 13.0, 31.0, 11.0, 17.0, 23.0, 32.0, 44.0, 68.0, 70.0, 66.0, 68.0, 65.0, 50.0] [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 10.0, 30.0, 10.0, 10.0, 10.0, 0.0]
[27]
s [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
c [22, 23, 24, 25, 26]
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
Set parameter TimeLimit to value 100
Status:2
Objective:2072.6771003651734
CPU time:1.7961080074310303
{0: [0, 22, 23, 25, 24, 1, 27], 2: [0, 26, 27]}
Gurobi solution: The total distance : 72.68
CPU time:1.8 seconds
DP list [22, 23, 25, 26] [24]
file name:  c101_21 backhaul number is 2:50%; 3:66%; 4